In [23]:
pip install PySastrawi

In [24]:
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
import pandas as pd
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from collections import Counter

In [17]:
df = pd.read_excel("dataset tugas feature extraction.xlsx")
tweets = df['full_text'].astype(str)
tweets

,full_text
0,@witasyahila @Hnirankara Anda layak jadi Owner...
1,Aldis Burger &gt;&gt;&gt; MBG
2,Kira2 Apa Komentar Menteri Hukum Prof Yusril t...
3,Iya sangat terasa sekali gak ada makanan bergi...
4,@Heraloebss MBG &gt;&gt;&gt; Mendem Bareng Gaess
...,...
195,@Hnirankara lebih busuk.pemilik mbg sppg memen...
196,Ceo mbg : Selamat anda kena prengg kameranya d...
197,@PUTRY_NUSANTARA Penyebetan stunting dengan st...
198,@SupaShorty A nigga quoted my tweet saying bae...


In [18]:
def clean_text(text):
    text = text.lower()          # ubah menjadi huruf kecil semua
    text = re.sub(r'http\S+', '', text)  # hapus URL
    text = re.sub(r'@\w+', '', text)     # hapus mention
    text = re.sub(r'#\w+', '', text)     # hapus hashtag
    text = re.sub(r'[^\w\s]', '', text)  # hapus tanda baca
    return text

In [19]:
stop_words = set(stopwords.words('indonesian'))

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def preprocess(text):
    text = clean_text(text)
    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]

    return tokens

In [25]:
processed = tweets.apply(preprocess)
processed

,full_text
0,"[layak, owner, mbg]"
1,"[aldis, burger, gtgtgt, mbg]"
2,"[kira2, komentar, menteri, hukum, prof, yusril..."
3,"[iya, gak, makan, gizi, mbg, libur]"
4,"[mbg, gtgtgt, mendem, bareng, gaess]"
...,...
195,"[busukpemilik, mbg, sppg, penting, gasil, mutu..."
196,"[ceo, mbg, selamat, kena, prengg, kamera, sana]"
197,"[sebet, stunting, stanting, aja, tanda, masyar..."
198,"[a, nigga, quoted, my, tweet, saying, baeboy, ..."


In [28]:
# ubah ke string lagi
processed_text = processed.apply(lambda x: ' '.join(x))
processed_text



,full_text
0,layak owner mbg
1,aldis burger gtgtgt mbg
2,kira2 komentar menteri hukum prof yusril carut...
3,iya gak makan gizi mbg libur
4,mbg gtgtgt mendem bareng gaess
...,...
195,busukpemilik mbg sppg penting gasil mutu kasih...
196,ceo mbg selamat kena prengg kamera sana
197,sebet stunting stanting aja tanda masyarakat b...
198,a nigga quoted my tweet saying baeboy i never ...


In [29]:
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(processed_text)
X_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1782 stored elements and shape (200, 1038)>

In [30]:
word_counts = X_bow.toarray().sum(axis=0)
words = vectorizer.get_feature_names_out()

freq_dict = dict(zip(words, word_counts))
top10_bow = Counter(freq_dict).most_common(10)

print("Top 10 Kata (BoW):")
for word, freq in top10_bow:
    print(word, ":", freq)

Top 10 Kata (BoW):
mbg : 175
yg : 34
ya : 24
rakyat : 14
dapur : 13
makan : 13
aja : 12
program : 12
efisiensi : 11
ga : 11


In [31]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(processed_text)

tfidf_scores = X_tfidf.toarray().sum(axis=0)
words = tfidf.get_feature_names_out()

tfidf_dict = dict(zip(words, tfidf_scores))
top10_tfidf = Counter(tfidf_dict).most_common(10)

print("\nTF IDF dari Top 10 Kata:")
for word, score in top10_tfidf:
    print(word, ":", score)


TF IDF dari Top 10 Kata:
mbg : 19.55959887468507
efisiensi : 5.742404291068905
sesuai : 5.4190683287670565
yg : 5.314832398500059
ya : 4.524924715158475
makan : 3.542190472805033
my : 3.069433563563289
rakyat : 2.8023511339724076
gak : 2.738877630032215
bgt : 2.707848148909166


**N GRAM**

Bigram

In [32]:
bigram_vectorizer = CountVectorizer(ngram_range=(2,2))
X_bigram = bigram_vectorizer.fit_transform(processed_text)

bigram_counts = X_bigram.toarray().sum(axis=0)
bigrams = bigram_vectorizer.get_feature_names_out()

bigram_dict = dict(zip(bigrams, bigram_counts))
top10_bigram = Counter(bigram_dict).most_common(10)

print("\nTop 10 Bigram:")
for bg, freq in top10_bigram:
    print(bg, ":", freq)


Top 10 Bigram:
dapur mbg : 11
efisiensi mbg : 9
mbg sesuai : 8
sesuai mbg : 8
ceo mbg : 7
program mbg : 6
mbg yg : 4
proyek mbg : 4
didik sehat : 3
jju ngaku : 3


Trigram

In [33]:
trigram_vectorizer = CountVectorizer(ngram_range=(3,3))
X_trigram = trigram_vectorizer.fit_transform(processed_text)

trigram_counts = X_trigram.toarray().sum(axis=0)
trigrams = trigram_vectorizer.get_feature_names_out()

trigram_dict = dict(zip(trigrams, trigram_counts))
top10_trigram = Counter(trigram_dict).most_common(10)

print("\nTop 10 Trigram:")
for tg, freq in top10_trigram:
    print(tg, ":", freq)


Top 10 Trigram:
efisiensi mbg sesuai : 8
mbg sesuai mbg : 8
hadir resmi dapur : 2
hemat rp40 triliun : 2
jju ngaku hayabusa : 2
makan gizi gratis : 2
ranny fahd arafiq : 2
0732sleman hadir resmi : 1
10 balap jembatan : 1
10 mbg ya : 1
